In [1]:
import os
print(os.listdir("../data_raw"))

['metro_passengers_trips_by_station_monthly_2026-04-10_14-26-01_0001.csv']


In [3]:
import pandas as pd

df = pd.read_csv("../data_raw/metro_passengers_trips_by_station_monthly_2026-04-10_14-26-01_0001.csv")
print(df.shape)
print(df.dtypes)
print(df['metro_station'].nunique(), "stations")
print(df.isna().sum())

(33776, 5)
metro_station       str
month               str
trips             int64
year              int64
load_timestamp      str
dtype: object
57 stations
metro_station     0
month             0
trips             0
year              0
load_timestamp    0
dtype: int64


In [4]:
month_map = {'Jan':1,'Feb':2,'Mar':3,'Apr':4,'May':5,'Jun':6,
             'Jul':7,'Aug':8,'Sep':9,'Oct':10,'Nov':11,'Dec':12}

df['metro_station'] = df['metro_station'].str.replace(r'\s+', ' ', regex=True).str.strip()
df['month_num'] = df['month'].map(month_map)
df['date'] = pd.to_datetime(dict(year=df['year'], month=df['month_num'], day=1))
df = df.sort_values(['metro_station', 'date']).reset_index(drop=True)

print(df['month'].unique())
print(df[['metro_station','month','year','date']].head())

<StringArray>
['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov',
 'Dec']
Length: 12, dtype: str
        metro_station month  year       date
0  ADCB Metro Station   Jan  2018 2018-01-01
1  ADCB Metro Station   Feb  2018 2018-02-01
2  ADCB Metro Station   Mar  2018 2018-03-01
3  ADCB Metro Station   Apr  2018 2018-04-01
4  ADCB Metro Station   May  2018 2018-05-01


In [5]:
dupes = df.duplicated(subset=['metro_station','date']).sum()
print("Duplicate station-month rows:", dupes)

gap_report = {}
for station, g in df.groupby('metro_station'):
    full_range = pd.date_range(g['date'].min(), g['date'].max(), freq='MS')
    missing = full_range.difference(g['date'])
    if len(missing) > 0:
        gap_report[station] = len(missing)

print(f"\nStations with gaps: {len(gap_report)} out of {df['metro_station'].nunique()}")
print(gap_report)

Duplicate station-month rows: 28834

Stations with gaps: 55 out of 57
{'ADCB Metro Station': 3, 'Abu Baker Al Siddique Metro Station': 3, 'Abu Hail Metro Station': 3, 'Airport Terminal 1 Metro Station': 3, 'Airport Terminal 3 Metro Station': 3, 'Al Fahidi Metro Station': 3, 'Al Furjan Metro Station': 5, 'Al Ghubaiba Metro Station': 3, 'Al Jadaf Metro Station': 3, 'Al Jafiliya Metro Station': 3, 'Al Nahda Metro Station': 3, 'Al Qiyadah Metro Station': 3, 'Al Qusais Metro Station': 3, 'Al Ras Metro Station': 3, 'Al Rigga Metro Station': 3, 'Baniyas Square Metro Station': 3, 'BurJuman Metro Station': 3, 'Burj Khalifa/ Dubai Mall Metro Station': 3, 'Business Bay Metro Station': 3, 'Creek Metro Station': 3, 'DAMAC Properties Metro Station': 3, 'DMCC Metro Station': 3, 'Danube Metro Station': 3, 'Deira City Centre Metro Station': 3, 'Discovery Gardens Metro Station': 7, 'Dubai Airport Free Zone Metro Station': 3, 'Dubai Healthcare City Metro Station': 3, 'Dubai Internet City Metro Station': 

In [6]:
# Look at one duplicated station-month in detail
sample = df[(df['metro_station']=='ADCB Metro Station') & (df['date']=='2018-01-01')]
print(sample)

# How many unique load_timestamp values exist overall?
print("\nUnique load_timestamps:", df['load_timestamp'].nunique())
print(df['load_timestamp'].value_counts().head(10))

        metro_station month   trips  year            load_timestamp  \
0  ADCB Metro Station   Jan  576653  2018  2018-03-07T15:45:02.000Z   

   month_num       date  
0          1 2018-01-01  

Unique load_timestamps: 613
load_timestamp
2026-03-26T15:42:08.000Z    1166
2022-02-08T23:39:30.000Z     159
2021-08-30T23:01:13.000Z     106
2024-02-28T12:07:41.000Z     106
2024-03-08T15:34:09.000Z     106
2024-04-01T19:54:08.000Z     106
2025-06-29T20:16:58.000Z     106
2025-07-09T12:42:46.000Z     106
2024-06-03T20:36:22.000Z      54
2020-09-03T17:32:03.000Z      53
Name: count, dtype: int64


In [7]:
# Find station-months that actually have more than 1 row
dup_mask = df.duplicated(subset=['metro_station','date'], keep=False)
dup_rows = df[dup_mask].sort_values(['metro_station','date'])
print(dup_rows.shape)
print(dup_rows.head(10))

# Check: for duplicated station-months, do trip values agree or differ?
agreement = dup_rows.groupby(['metro_station','date'])['trips'].nunique()
print("\nDuplicated groups where trips VALUE differs:", (agreement > 1).sum())
print("Duplicated groups where trips VALUE is identical:", (agreement == 1).sum())

(30319, 7)
         metro_station month    trips  year            load_timestamp  \
42  ADCB Metro Station   Jul   442492  2021  2021-08-30T23:01:13.000Z   
43  ADCB Metro Station   Jul   442492  2021  2021-08-30T23:01:13.000Z   
48  ADCB Metro Station   Dec   701038  2021  2022-02-08T23:39:30.000Z   
49  ADCB Metro Station   Dec   701038  2021  2022-02-08T23:39:30.000Z   
50  ADCB Metro Station   Dec   701038  2021  2022-02-08T23:39:30.000Z   
70  ADCB Metro Station   Aug   823153  2023  2023-12-04T20:11:59.000Z   
71  ADCB Metro Station   Aug   823153  2023  2023-10-02T20:31:40.000Z   
72  ADCB Metro Station   Dec   975685  2023  2024-02-28T12:07:41.000Z   
73  ADCB Metro Station   Dec   359828  2023  2024-02-28T12:07:41.000Z   
74  ADCB Metro Station   Jan  1025278  2024  2024-03-08T15:34:09.000Z   

    month_num       date  
42          7 2021-07-01  
43          7 2021-07-01  
48         12 2021-12-01  
49         12 2021-12-01  
50         12 2021-12-01  
70          8 2023-08-0

In [8]:
# Look at row order vs load_timestamp for a conflicting group
conflict_example = df[(df['metro_station']=='ADCB Metro Station') & (df['date']=='2023-12-01')]
print(conflict_example[['trips','load_timestamp']])
print("\nOriginal row index order:", conflict_example.index.tolist())

# Check: within groups with same load_timestamp but different trips, 
# is the LAST row in file order usually the larger or smaller value?

     trips            load_timestamp
72  975685  2024-02-28T12:07:41.000Z
73  359828  2024-02-28T12:07:41.000Z

Original row index order: [72, 73]


In [9]:
conflicting = dup_rows.groupby(['metro_station','date']).filter(lambda g: g['trips'].nunique() > 1)

def timestamp_varies(g):
    return g['load_timestamp'].nunique() > 1

resolvable_by_recency = conflicting.groupby(['metro_station','date']).filter(timestamp_varies)
same_timestamp_conflict = conflicting.groupby(['metro_station','date']).filter(lambda g: not timestamp_varies(g))

n_resolvable = resolvable_by_recency.groupby(['metro_station','date']).ngroups
n_unresolvable = same_timestamp_conflict.groupby(['metro_station','date']).ngroups

print("Conflicting groups resolvable by most-recent load_timestamp:", n_resolvable)
print("Conflicting groups with SAME timestamp but different trips (unresolvable by recency):", n_unresolvable)

Conflicting groups resolvable by most-recent load_timestamp: 1113
Conflicting groups with SAME timestamp but different trips (unresolvable by recency): 53


In [10]:
summary = same_timestamp_conflict.groupby(['metro_station','date'])['trips'].agg(['min','max','count'])
summary['pct_diff'] = (summary['max'] - summary['min']) / summary['max'] * 100
print(summary.sort_values('pct_diff', ascending=False).head(15))
print("\nMedian % difference:", summary['pct_diff'].median().round(1))

                                                   min      max  count  \
metro_station                       date                                 
EXPO Metro Station                  2023-12-01   75920   522667      2   
Jumeirah Golf Estates Metro Station 2023-12-01   10219    32927      2   
Nakheel Metro Station               2023-12-01   61557   184323      2   
World Trade Centre Metro Station    2023-12-01  200640   593092      2   
Energy Metro Station                2023-12-01   15532    45646      2   
DMCC Metro Station                  2023-12-01  350230  1014768      2   
First Abu Dhabi Bank Metro Station  2023-12-01  156235   452676      2   
Dubai Investment Park Metro Station 2023-12-01  114602   330983      2   
Oud Metha Metro Station             2023-12-01  218050   627872      2   
Jabal Ali Metro Station             2023-12-01   21141    60748      2   
Baniyas Square Metro Station        2023-12-01  345169   984514      2   
Dubai Internet City Metro Station   20

In [11]:
print("All unique dates among the 53 unresolvable conflicts:", 
      same_timestamp_conflict['date'].unique())

summary['ratio'] = summary['max'] / summary['min']
print("\nRatio (max/min) stats:")
print(summary['ratio'].describe())

All unique dates among the 53 unresolvable conflicts: <DatetimeArray>
['2023-12-01 00:00:00']
Length: 1, dtype: datetime64[us]

Ratio (max/min) stats:
count    53.000000
mean      2.871244
std       0.569915
min       2.670804
25%       2.721199
50%       2.783846
75%       2.841672
max       6.884444
Name: ratio, dtype: float64


In [12]:
# Step 1: resolve the 1,113 groups by keeping the most recent load_timestamp
df_sorted = df.sort_values('load_timestamp')
df_deduped = df_sorted.drop_duplicates(subset=['metro_station','date'], keep='last')

# Step 2: exclude the 53 unresolvable Dec 2023 conflicts entirely
unresolvable_keys = set(zip(same_timestamp_conflict['metro_station'], same_timestamp_conflict['date']))
mask_exclude = df_deduped.apply(lambda r: (r['metro_station'], r['date']) in unresolvable_keys, axis=1)
df_clean = df_deduped[~mask_exclude].copy()

df_clean = df_clean.sort_values(['metro_station','date']).reset_index(drop=True)

print("Original rows:", len(df))
print("After cleaning:", len(df_clean))
print("Remaining duplicates:", df_clean.duplicated(subset=['metro_station','date']).sum())

Original rows: 33776
After cleaning: 4889
Remaining duplicates: 0


In [13]:
print(df_clean.shape)
print("Stations:", df_clean['metro_station'].nunique())
print("Date range:", df_clean['date'].min(), "to", df_clean['date'].max())
print("Avg months per station:", round(len(df_clean) / df_clean['metro_station'].nunique(), 1))

(4889, 7)
Stations: 57
Date range: 2018-01-01 00:00:00 to 2026-04-01 00:00:00
Avg months per station: 85.8


In [14]:
df_clean.to_csv("../data_processed/metro_trips_clean.csv", index=False)
print("Saved:", df_clean.shape)

Saved: (4889, 7)


In [15]:
station_ranges = df_clean.groupby('metro_station').agg(
    first_month=('date', 'min'),
    last_month=('date', 'max'),
    n_records=('date', 'count')
).sort_values('first_month')

pd.set_option('display.max_rows', None)
print(station_ranges)

                                       first_month last_month  n_records
metro_station                                                           
ADCB Metro Station                      2018-01-01 2026-04-01         96
Dubai Internet City Metro Station       2018-01-01 2026-04-01         96
Union Metro Station                     2018-01-01 2026-04-01         96
Emirates Metro Station                  2018-01-01 2026-04-01         96
Emirates Towers Metro Station           2018-01-01 2026-04-01         96
Energy Metro Station                    2018-01-01 2026-04-01         96
Etisalat Metro Station                  2018-01-01 2026-04-01         96
Financial Centre Metro Station          2018-01-01 2026-04-01         96
First Abu Dhabi Bank Metro Station      2018-01-01 2026-04-01         96
GGICO Metro Station                     2018-01-01 2026-04-01         96
Ibn Battuta Metro Station               2018-01-01 2026-04-01         96
Dubai Healthcare City Metro Station     2018-01-01 

In [16]:
candidates = ['Nakheel Harbour & Tower Metro Station', 'Jabal Ali Metro Station',
              'Sharaf DG Metro Station', 'Mashreq Metro Station',
              'Palm Deira Metro Station', 'Gold Souq Metro Station',
              'Jumeirah Lakes Towers Metro Station', 'DMCC Metro Station']

check = df_clean[df_clean['metro_station'].isin(candidates)]
print(check.groupby('metro_station').agg(first=('date','min'), last=('date','max'), n=('date','count')))

                                           first       last   n
metro_station                                                  
DMCC Metro Station                    2018-05-01 2026-04-01  92
Gold Souq Metro Station               2020-02-01 2026-04-01  70
Jabal Ali Metro Station               2020-02-01 2026-04-01  69
Jumeirah Lakes Towers Metro Station   2018-01-01 2018-04-01   4
Mashreq Metro Station                 2020-04-01 2026-04-01  69
Nakheel Harbour & Tower Metro Station 2018-01-01 2020-03-01   2
Palm Deira Metro Station              2018-01-01 2020-03-01  26
Sharaf DG Metro Station               2018-01-01 2020-03-01  27


In [17]:
rename_map = {
    'Jumeirah Lakes Towers Metro Station': 'DMCC Metro Station',
    'Sharaf DG Metro Station': 'Mashreq Metro Station',
    'Nakheel Harbour & Tower Metro Station': 'Jabal Ali Metro Station',
    'Palm Deira Metro Station': 'Gold Souq Metro Station',
}

df_clean['metro_station'] = df_clean['metro_station'].replace(rename_map)

# Re-check for duplicates created by the merge (from the overlap months)
overlap_dupes = df_clean.duplicated(subset=['metro_station','date'], keep=False)
print("Rows involved in new overlap duplicates:", overlap_dupes.sum())
print(df_clean[overlap_dupes].sort_values(['metro_station','date']))

Rows involved in new overlap duplicates: 0
Empty DataFrame
Columns: [metro_station, month, trips, year, load_timestamp, month_num, date]
Index: []


In [18]:
print("Stations after rename merge:", df_clean['metro_station'].nunique())
print("Total rows:", len(df_clean))
print("Any remaining duplicates:", df_clean.duplicated(subset=['metro_station','date']).sum())

Stations after rename merge: 53
Total rows: 4889
Any remaining duplicates: 0


In [19]:
df_clean.to_csv("../data_processed/metro_trips_clean.csv", index=False)
print("Saved:", df_clean.shape)

Saved: (4889, 7)


In [20]:
station_list = sorted(df_clean['metro_station'].unique())
for s in station_list:
    print(s)

ADCB Metro Station
Abu Baker Al Siddique Metro Station
Abu Hail Metro Station
Airport Terminal 1 Metro Station
Airport Terminal 3 Metro Station
Al Fahidi Metro Station
Al Furjan Metro Station
Al Ghubaiba Metro Station
Al Jadaf Metro Station
Al Jafiliya Metro Station
Al Nahda Metro Station
Al Qiyadah Metro Station
Al Qusais Metro Station
Al Ras Metro Station
Al Rigga Metro Station
Baniyas Square Metro Station
BurJuman Metro Station
Burj Khalifa/ Dubai Mall Metro Station
Business Bay Metro Station
Creek Metro Station
DAMAC Properties Metro Station
DMCC Metro Station
Danube Metro Station
Deira City Centre Metro Station
Discovery Gardens Metro Station
Dubai Airport Free Zone Metro Station
Dubai Healthcare City Metro Station
Dubai Internet City Metro Station
Dubai Investment Park Metro Station
EXPO Metro Station
Emirates Metro Station
Emirates Towers Metro Station
Energy Metro Station
Etisalat Metro Station
Financial Centre Metro Station
First Abu Dhabi Bank Metro Station
GGICO Metro Statio

In [21]:
print(df_clean['trips'].describe())
print("\nZero or negative trips:", (df_clean['trips'] <= 0).sum())

count    4.889000e+03
mean     2.646044e+05
std      3.032190e+05
min      1.000000e+00
25%      2.391200e+04
50%      1.669430e+05
75%      4.066350e+05
max      2.275405e+06
Name: trips, dtype: float64

Zero or negative trips: 0


In [22]:
import re

for s in station_list:
    # Flag anything with double spaces, odd punctuation spacing, or non-standard characters
    if re.search(r'\s{2,}', s) or re.search(r'/\S', s) or re.search(r'\S/', s) or re.search(r'[^\w\s&/\'\.\-]', s):
        print(repr(s))

'Burj Khalifa/ Dubai Mall Metro Station'


In [23]:
low_trips = df_clean[df_clean['trips'] < 100].sort_values('trips')
print(low_trips[['metro_station','date','trips']])

                            metro_station       date  trips
2625  Dubai Investment Park Metro Station 2020-03-01      1
2273      Discovery Gardens Metro Station 2020-05-01      1
3678  Jumeirah Golf Estates Metro Station 2021-04-01      1
4538            The Gardens Metro Station 2020-09-01      1
3672  Jumeirah Golf Estates Metro Station 2020-09-01      1
3677  Jumeirah Golf Estates Metro Station 2021-03-01      1
3671  Jumeirah Golf Estates Metro Station 2020-08-01      1
576               Al Furjan Metro Station 2020-02-01      2
4539            The Gardens Metro Station 2020-11-01      2
2627  Dubai Investment Park Metro Station 2020-08-01      2
582               Al Furjan Metro Station 2020-09-01      3
3670  Jumeirah Golf Estates Metro Station 2020-07-01      4
3604              Jabal Ali Metro Station 2020-11-01      4
581               Al Furjan Metro Station 2020-07-01      4
2633  Dubai Investment Park Metro Station 2021-03-01      5
2691                   EXPO Metro Statio

In [24]:
df_clean['metro_station'] = df_clean['metro_station'].str.replace(
    'Burj Khalifa/ Dubai Mall Metro Station', 'Burj Khalifa/Dubai Mall Metro Station'
)
print(df_clean['metro_station'].unique()[df_clean['metro_station'].str.contains('Burj Khalifa').values])

IndexError: Boolean index has wrong length: 4889 instead of 53

In [25]:
matches = [s for s in df_clean['metro_station'].unique() if 'Burj Khalifa' in s]
print(matches)

['Burj Khalifa/Dubai Mall Metro Station']


In [26]:
print("Rows:", len(df_clean))
print("Stations:", df_clean['metro_station'].nunique())

Rows: 4889
Stations: 53


In [27]:
print("Rows:", len(df_clean))
print("Stations:", df_clean['metro_station'].nunique())

df_clean.to_csv("../data_processed/metro_trips_clean.csv", index=False)
print("Saved:", df_clean.shape)

Rows: 4889
Stations: 53
Saved: (4889, 7)
